In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [3]:
df = spark.read.json("cwiczenia/RTA/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
#zadanie 2.1
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [6]:
#zadanie 2.2
from pyspark.sql.functions import min as _min, max as _max

category_stats = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    )
    .orderBy("category")
)

category_stats.show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [12]:
#zadanie 3.1
from pyspark.sql.functions import window, col,desc
hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [10]:
#zadanie 3.2
windows_30m = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od", "store")
)

windows_30m.show(truncate=False)

+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [13]:
#zadanie 3.3
krakow_top = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "suma_PLN",
    )
    .orderBy(desc("suma_PLN"))
)

krakow_top.show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |suma_PLN |
+-------------------+-------------------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|201259.26|
+-------------------+-------------------+---------+



In [14]:
#zadanie 4.1
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [16]:
#zadanie 4.2
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# ponieważ okna nakładają się na siebie, jedna transakcja może należeć do kilku okien jednocześnie


Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [17]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") grupuje dane tylko według sklepu a groupBy(window(...), "store") grupuje dane według czasu i sklepu jednocześnie

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna ( 9:00-10:00 i 9:30-10:30)

In [19]:
#praca domowa 1
gdansk_lowest = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_PLN",
    )
    .orderBy("srednia_PLN")
)

gdansk_lowest.show(truncate=False)

# najniższa średnia kwota transakcji była miedzy 8:00 a 9:00

+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
|2026-04-12 10:00:00|2026-04-12 11:00:00|412.92     |
|2026-04-12 09:00:00|2026-04-12 10:00:00|415.91     |
+-------------------+-------------------+-----------+



In [22]:
# praca domowa 2
transactions_morning = (
    df.filter(
        (col("timestamp") >= "2026-04-12 09:00:00") &
        (col("timestamp") < "2026-04-12 09:30:00")
    )
    .groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .orderBy(desc("liczba_tx"))
)

transactions_morning.show()

+-----------+---------+
|   category|liczba_tx|
+-----------+---------+
|    książki|      622|
|elektronika|      611|
|     odzież|      605|
|    żywność|      567|
+-----------+---------+



In [23]:
# praca domowa 3
peak_15m = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
    )
    .orderBy(desc("liczba_tx"))
)

peak_15m.show(truncate=False)

# szczyt transakcji był między 9:15 a 9:30

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+



In [24]:
spark.stop()